In [1]:
import uproot
import awkward as ak
import numpy as np

# ==============================================================================
# 1. FUNÇÃO DE TRUTH MATCHING (B0 -> K*0 J/psi)
# ==============================================================================
def apply_b0_jpsi_truth_match(pdg, mother, **branches):
    local_idx = ak.local_index(pdg)

    # --- MAPEAMENTO ---
    safe_mom = ak.where(mother >= 0, mother, 0)
    mom_pdg  = pdg[safe_mom]
    mom_pdg  = ak.where(mother >= 0, mom_pdg, -999)

    safe_grandmom = safe_mom[safe_mom]
    grandmom_pdg  = pdg[safe_grandmom]
    grandmom_pdg  = ak.where(safe_mom >= 0, grandmom_pdg, -999)

    safe_greatgrandmom = safe_mom[safe_grandmom]
    greatgrandmom_pdg  = pdg[safe_greatgrandmom]
    greatgrandmom_pdg  = ak.where(safe_grandmom >= 0, greatgrandmom_pdg, -999)

    # --- RASTREAMENTO ---
    B_ancestor_idx = ak.full_like(pdg, -1)
    B_ancestor_idx = ak.where((pdg == 313) & (mom_pdg == 511), safe_mom, B_ancestor_idx)
    B_ancestor_idx = ak.where((pdg == 443) & (mom_pdg == 511), safe_mom, B_ancestor_idx)

    is_K  = (pdg == 321)  & (mom_pdg == 313) & (grandmom_pdg == 511)
    is_pi = (pdg == -211) & (mom_pdg == 313) & (grandmom_pdg == 511)
    B_ancestor_idx = ak.where(is_K, safe_grandmom, B_ancestor_idx)
    B_ancestor_idx = ak.where(is_pi, safe_grandmom, B_ancestor_idx)
    
    is_photon_from_jpsi = (pdg == 22) & (mom_pdg == 443) & (grandmom_pdg == 511)
    B_ancestor_idx = ak.where(is_photon_from_jpsi, safe_grandmom, B_ancestor_idx)
    
    is_muon_direct = (np.abs(pdg) == 13) & (mom_pdg == 443) & (grandmom_pdg == 511)
    B_ancestor_idx = ak.where(is_muon_direct, safe_grandmom, B_ancestor_idx)

    is_muon_via_photon = (np.abs(pdg) == 13) & (mom_pdg == 22) & (grandmom_pdg == 443) & (greatgrandmom_pdg == 511)
    B_ancestor_idx = ak.where(is_muon_via_photon, safe_greatgrandmom, B_ancestor_idx)

    B_ancestor_idx = ak.where((pdg == 511), local_idx, B_ancestor_idx)

    # --- FILTRAGEM FINAL ABSOLUTA (Regra do RG) ---
    jpsi_ancestors = B_ancestor_idx[pdg == 443]
    signal_b_idx = ak.fill_none(ak.firsts(jpsi_ancestors), -99) 
    
    # A Máscara de Ferro
    particle_mask = (B_ancestor_idx == signal_b_idx) & (B_ancestor_idx >= 0)
    pdg_filtered = pdg[particle_mask]
    
    has_jpsi  = ak.sum(pdg_filtered == 443, axis=1) > 0
    has_muons = ak.sum(np.abs(pdg_filtered) == 13, axis=1) >= 2
    is_valid_signal = has_jpsi & has_muons
    
    results = {'pdg': pdg_filtered[is_valid_signal]}
    for name, array in branches.items():
        results[name] = array[particle_mask][is_valid_signal]
        
    return results


# ==============================================================================
# 2. SCRIPT PRINCIPAL
# ==============================================================================
file_path = "/eos/user/t/tdeandra/skim_outputs/Merged_Eras/Merged_mc_jpsi_2022.root"

# Usando o bloco 'with' para garantir o fechamento correto do arquivo ROOT
with uproot.open(file_path) as file:
    tree = file["Events"]

    # Extrai os arrays brutos
    raw_pdg    = tree["BPHGenPart_pdgId"].array()
    raw_mother = tree["BPHGenPart_genPartIdxMother"].array()
    raw_mass   = tree["BPHGenPart_mass"].array()
    # raw_pt     = tree["BPHGenPart_pt"].array() # Adicione as variáveis cinemáticas aqui

# Executa o truth match
filtered_data = apply_b0_jpsi_truth_match(
    pdg=raw_pdg, 
    mother=raw_mother, 
    mass=raw_mass
    # pt=raw_pt
)

# Separa os resultados
pdg_clean  = filtered_data['pdg']
mass_clean = filtered_data['mass']

# ==============================================================================
# 3. VERIFICAÇÃO DOS RESULTADOS
# ==============================================================================
print(f"--- RESUMO DA FILTRAGEM ---")
print(f"Total de eventos processados : {len(raw_pdg)}")
print(f"Total de eventos de SINAL    : {len(pdg_clean)}\n")

print("--- INSPECIONANDO OS 5 PRIMEIROS EVENTOS VÁLIDOS ---")
for i in range(min(5, len(pdg_clean))):
    print(f"Evento {i}:")
    print(f"  PDG  : {pdg_clean[i].tolist()}")
    print(f"  Mass : {mass_clean[i].tolist()}\n")

--- RESUMO DA FILTRAGEM ---
Total de eventos processados : 436465
Total de eventos de SINAL    : 215967

--- INSPECIONANDO OS 5 PRIMEIROS EVENTOS VÁLIDOS ---
Evento 0:
  PDG  : [511, 443, -13, 22, 13, 22, 22, 22, 22, 313, 321, -211]
  Mass : [5.28125, 3.09375, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.912109375, 0.494140625, 0.0]

Evento 1:
  PDG  : [511, 443, -13, 22, 13, 22, 313, 321, -211]
  Mass : [5.28125, 3.09375, 0.0, 0.0, 0.0, 0.0, 0.873046875, 0.494140625, 0.0]

Evento 2:
  PDG  : [511, 443, -13, 22, 13, 22, 313, 321, -211]
  Mass : [5.28125, 3.09375, 0.0, 0.0, 0.0, 0.0, 0.939453125, 0.494140625, 0.0]

Evento 3:
  PDG  : [511, 443, -13, 22, 13, 22, 313, 321, -211]
  Mass : [5.28125, 3.09375, 0.0, 0.0, 0.0, 0.0, 0.98046875, 0.494140625, 0.0]

Evento 4:
  PDG  : [511, 443, -13, 22, 13, 22, 313, 321, -211]
  Mass : [5.28125, 3.09375, 0.0, 0.0, 0.0, 0.0, 0.9296875, 0.494140625, 0.0]

